In [14]:
from openai import OpenAI, pydantic_function_tool
from pydantic import BaseModel, Field

In [15]:
MODEL_ID="llama3.1:8b"
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

In [16]:
class GetWeather(BaseModel):
    """Get the current weather for one city."""
    city: str = Field(description="City name, e.g Paris")

The class name becomes tool name, the docstring and description 
tell the model when
and how to use it

In [17]:
def get_weather(args: GetWeather) -> str:
    fake = {
        "Paris": "18C, rainy...",
        "Tokyo": "25C, shiny...",
        "Chicago": "30C, shiny..."
    }
    return fake.get(args.city, "No data for this city...")

In [18]:
tools = [pydantic_function_tool(GetWeather)]
messages = [{"role": "user", "content": "Weather in Bishkek"}]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    tools=tools
)
assistant = response.choices[0].message
messages.append(assistant)

In [19]:
for call in assistant.tool_calls or []:
    args = GetWeather.model_validate_json(call.function.arguments)
    result = get_weather(args)
    messages.append({
        "role": "tool", "tool_call_id": call.id, "content": result
    })
    
final = client.chat.completions.create(
    model=MODEL_ID, reasoning_effort="none", messages=messages, tools=tools
)
print(final.choices[0].message.content)

It seems like the city "Bishkek" is not supported. Let me try again with a different city. 

However, Bishkek is the capital of Kyrgyzstan. It has a continental climate with significant temperature variations between summer and winter. In the summer, temperatures can rise up to 32°C (90°F), while in the winter, they can drop as low as -20°C (-4°F).
